# LungCT All-10 Pareto Visual Review

This notebook combines IDIR BE, DirectCP 4stage pTV-LCC+isoTV, DirectCP 4stage global-NCC+FD-BE, and uploaded pTVReg fields. It displays the screening table and saves all Pareto figures to disk.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.reports.lungct_pareto_visual_review import (
    DEFAULT_CASE_ROWS,
    DEFAULT_OUTPUT_ROOT,
    DEFAULT_PTVPREP_ROOT,
    DEFAULT_PTVREG_ROOT,
    METHOD_FDBE,
    METHOD_IDIR,
    METHOD_PTVREG,
    METHOD_PTV_DIRECTCP,
    add_ratios,
    aggregate_case_rows,
    load_method_case_rows,
    load_ptvreg_case_rows,
    make_all_plots,
    save_review_artifacts,
    screening_table,
    validate_outputs,
)

pd.set_option('display.max_rows', 80)
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
print(f'Repo root: {REPO_ROOT}')
print(f'Output root: {DEFAULT_OUTPUT_ROOT}')

## Editable Exclusion Controls

Edit the dictionaries below to exclude by ratio-to-best-TRE weight or exact regularization weight. pTVReg is always retained as the reference point.

In [ ]:
OUTPUT_ROOT = DEFAULT_OUTPUT_ROOT

EXCLUDE_BY_RATIO = {
    METHOD_IDIR: [],
    METHOD_PTV_DIRECTCP: [10.0, 100.0],
    METHOD_FDBE: [0.0, 0.001],
}

EXCLUDE_BY_WEIGHT = {
    METHOD_IDIR: [],
    METHOD_PTV_DIRECTCP: [],
    METHOD_FDBE: [],
}

print('EXCLUDE_BY_RATIO =', EXCLUDE_BY_RATIO)
print('EXCLUDE_BY_WEIGHT =', EXCLUDE_BY_WEIGHT)

## Load Data And Validate pTVReg Fields

In [ ]:
method_case_df = load_method_case_rows(DEFAULT_CASE_ROWS)
ptvreg_case_df = load_ptvreg_case_rows(DEFAULT_PTVREG_ROOT, DEFAULT_PTVPREP_ROOT)
case_df = add_ratios(pd.concat([method_case_df, ptvreg_case_df], ignore_index=True, sort=False))
mean_df = aggregate_case_rows(case_df)
table_df = screening_table(mean_df)
artifact_paths = save_review_artifacts(case_df, mean_df, table_df, OUTPUT_ROOT)

validation_cols = [
    'case_id',
    'source_saved_tre_phys_mean_mm',
    'source_recomputed_snap_tre_mean_mm',
    'source_tre_validation_abs_err',
    'source_saved_trestd_case_mm',
    'source_recomputed_snap_tre_std_mm',
    'source_trestd_validation_abs_err',
    'source_disp_h5py_shape',
]
ptvreg_validation = ptvreg_case_df[validation_cols].copy()

print(f'Trained-method case rows: {len(method_case_df)}')
print(f'pTVReg case rows: {len(ptvreg_case_df)}')
print(f'Combined case rows: {len(case_df)}')
print(f'Combined mean rows: {len(mean_df)}')
print('Saved artifacts:')
for name, path in artifact_paths.items():
    print(f'  {name}: {path}')

display(Markdown('### pTVReg field validation'))
display(ptvreg_validation)
assert (ptvreg_validation['source_tre_validation_abs_err'] < 1e-8).all()
assert (ptvreg_validation['source_trestd_validation_abs_err'] < 1e-8).all()

## Screening Table

In [ ]:
display(table_df)

## Pareto Figures

Each ellipse is centered at the method/weight all-case mean. Width is one standard deviation on the regularity axis, and height is one standard deviation on original snap TRE.

In [ ]:
plot_paths = make_all_plots(mean_df, OUTPUT_ROOT, EXCLUDE_BY_RATIO, EXCLUDE_BY_WEIGHT)
display(Markdown('### Saved figure paths'))
for subset_name, paths in plot_paths.items():
    display(Markdown(f'**{subset_name}**'))
    for path in paths:
        print(path)

## Output Check

In [ ]:
status = validate_outputs(OUTPUT_ROOT)
display(status)
assert status['ok'], status